# Tutorial to train a CNN for pulsar population synthesis

In this tutorial we will train a convolutional neural network (CNN) on a dataset of simulated populations to learn
the means of the initial spin period and magnetic field distributions in log, i.e., `P_initial_log10_mean` and `B_initial_log10_mean`.

We use here a supervised learning approach where training data are simulated neutron star populations represented in the form of $P-\dot{P}$ diagram density maps which suitably labeled with the two parameters `P_initial_log10_mean` and `B_initial_log10_mean`. The network has to learn to predict the target value of the label parameters associated to the simulated population samples.
We use a three channel input formed by three $P-\dot{P}$ density maps resulting from three simulated radio survey.

To run this example, we use the following command:
```commandline
python pypopsyn/learning/train.py --configuration config_train.json
```
The `config_train.json` contains all the information necessary to set up the training experiment, such as the training and validation dataset to use, the network architecture, the optimization procedure and several others (see documentation for more details).
You can also customize your own configuration file and provide it to the train script.

In [ ]:
import collections
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pathlib
import sys

from matplotlib.ticker import ScalarFormatter

import utilities.plot_settings

## Run the training script

In [ ]:
!python ../../pypopsyn/learning/train.py --configuration config_train.json

## Extract training results

In the following we extract the training and validation losses for the two parameters we are predicting in order to see how they evolve over the training process.

In [ ]:
with open(f"config_train.json", "r") as read_file:
    config_train = json.load(read_file)

train_output_path = config_train["trainer"]["save_dir"]
log_output_path = f"{train_output_path}/logs/Convolution/20240726_111802/"
model_output_path = f"{train_output_path}/models/Convolution/20240726_111802/"
infer_output_path = config_train["infer"]["save_dir"]

In [ ]:
with open(f"{log_output_path}train_eval_result.json", "r") as read_file:
    train_data = json.load(read_file)
with open(f"{log_output_path}validation_result.json", "r") as read_file:
    valid_data = json.load(read_file)

In [ ]:
epochs = train_data.keys()
train_loss_P_initial_log10_mean = []
valid_loss_P_initial_log10_mean = []
train_loss_B_initial_log10_mean = []
valid_loss_B_initial_log10_mean = []

for e in epochs:
    train_loss_P_initial_log10_mean.append(train_data[e]["P_initial_log10_mean"])
    train_loss_B_initial_log10_mean.append(train_data[e]["B_initial_log10_mean"])
    valid_loss_P_initial_log10_mean.append(valid_data[e]["P_initial_log10_mean"])
    valid_loss_B_initial_log10_mean.append(valid_data[e]["B_initial_log10_mean"])

epochs = np.array(list(epochs), dtype="float64") 

Plot the training and validation losses as a function of the training epoch for the two parameters to predict.

In [ ]:
# plot loss trend P_initial_log10_mean
colors = ["tab:red", "tab:blue", "tab:orange", "tab:green"]

fig, ax = plt.subplots(figsize=(15,9))

ax.plot(
    epochs, 
    train_loss_P_initial_log10_mean, 
    linestyle="-",
    linewidth=3,
    color=colors[0],
    alpha=0.9,
    rasterized=True,
    label=r"Training" 
)
ax.plot(
    epochs, 
    valid_loss_P_initial_log10_mean, 
    linestyle="-",
    linewidth=3,
    color=colors[1],
    alpha=0.9,
    rasterized=True,
    label=r"Validation" 
)

ax.set_xlabel(r'Training epoch')
ax.set_ylabel(r'$\mu_{\log P}$ ($P$ in s) MSE loss')
ax.set_xscale('log')
ax.set_yscale('log')
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.legend(frameon=False, loc=0, fontsize=24)

In [ ]:
# plot loss trend B_initial_log10_mean
fig, ax = plt.subplots(figsize=(15,9))

ax.plot(
    epochs, 
    train_loss_B_initial_log10_mean, 
    linestyle="-",
    linewidth=3,
    color=colors[0],
    alpha=0.9,
    rasterized=True,
    label=r"Training" 
)
ax.plot(
    epochs, 
    valid_loss_B_initial_log10_mean, 
    linestyle="-",
    linewidth=3,
    color=colors[1],
    alpha=0.9,
    rasterized=True,
    label=r"Validation" 
)

ax.set_xlabel(r'Training epoch')
ax.set_ylabel(r'$\mu_{\log B}$ ($B$ in G) MSE loss')
ax.set_xscale('log')
ax.set_yscale('log')
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.legend(frameon=False, loc=0, fontsize=24)

plt.show()

## Perform inference with the trained model

Once the model has been trained we can use it to perform inference on any new input sample.
Here we show the performance of the model in predicting the two parameters on the validation dataset.

In [ ]:
# define the path to the trained model.
trained_model_path = f"{model_output_path}/best_model_trial1.pth"

Run the inference script.

In [ ]:
# Construct the command.
command = f"python ../../pypopsyn/learning/infer.py --configuration config_train.json --trained_model {trained_model_path}"

# Execute the command.
!{command}

Extract the inference results and plot them.

In [ ]:
data = pd.read_csv(
    f"{infer_output_path}/inference_results.csv"
)
target_P_initial_log10_mean = data["target:P_initial_log10_mean"].to_numpy()
prediction_P_initial_log10_mean = data["predicted:P_initial_log10_mean"].to_numpy()
target_B_initial_log10_mean = data["target:B_initial_log10_mean"].to_numpy()
prediction_B_initial_log10_mean = data["predicted:B_initial_log10_mean"].to_numpy()

In [ ]:
fig, ax = plt.subplots(figsize=(15, 8))

ax.set_xlabel(r"Target $\mu_{\log P}$ ($P$ in s)")
ax.set_ylabel(r"Predicted $\mu_{\log P}$ ($P$ in s)")

ax.scatter(
    target_P_initial_log10_mean,
    prediction_P_initial_log10_mean,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=50,
    alpha=1.0,
    rasterized=True,
)
ax.plot(
    target_P_initial_log10_mean,
    target_P_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color="tab:red",
    alpha=1.0,
    rasterized=True,
    zorder=1,
)

fig, ax = plt.subplots(figsize=(15, 8))

ax.set_xlabel(r"Target $\mu_{\log B}$ ($B$ in G)")
ax.set_ylabel(r"Predicted $\mu_{\log B}$ ($B$ in G)")

ax.scatter(
    target_B_initial_log10_mean,
    prediction_B_initial_log10_mean,
    linestyle="None",
    marker="o",
    facecolors="black",
    edgecolors="None",
    s=50,
    alpha=1.,
    rasterized=True,
)
ax.plot(
    target_B_initial_log10_mean,
    target_B_initial_log10_mean,
    linestyle="-",
    linewidth=3,
    color="tab:red",
    alpha=1.0,
    rasterized=True,
    zorder=1,
)
plt.show()